# Sliding Window Scalability


## 1. Imports

In [10]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Configuration


In [11]:
INPUT_NPZ        = "../Features/Features_AdaptiveDEFT_CoAttn_R50_v2/Feature_P01_01_AdaptiveDEFT_CoAttn_R50.npz"
RESULTS_FULL     = "results_phase3a_NewFeat_R50_full.csv"
RESULTS_SUMMARY  = "results_phase3a_NewFeat_R50_summary.csv"
PLOT_PATH        = "phase3a_NewFeat_R50_plots.png"

N_VALUES         = [30, 100, 150, 200]
STRIDE_RATIO     = 1.0
DPT_PERCENTILE   = 98
SEEDS            = [42, 123, 456]

NUM_EPOCHS       = 100
HIDDEN_DIM       = 128
HEADS            = 2
LR               = 1e-4
WEIGHT_DECAY     = 1e-4
BATCH_SIZE       = 16
MIN_PER_CLASS    = 2

print(f"INPUT_NPZ: {INPUT_NPZ}")
print(f"N values:  {N_VALUES}")
print(f"Seeds:     {SEEDS}")

INPUT_NPZ: ../Features/Features_AdaptiveDEFT_CoAttn_R50_v2/Feature_P01_01_AdaptiveDEFT_CoAttn_R50.npz
N values:  [30, 100, 150, 200]
Seeds:     [42, 123, 456]


## 3. Load Features

In [12]:
data = np.load(INPUT_NPZ, allow_pickle=True)
features      = data['features']
frame_numbers = data['frame_numbers']
clip_ids      = data['clip_ids']
action_labels = data['action_label']

print(f"Loaded: {INPUT_NPZ}")
print(f"Total frames: {features.shape[0]} | Feature dim: {features.shape[1]}")
print(f"Unique clips: {len(np.unique(clip_ids))} | Unique actions: {len(np.unique(action_labels))}")

Loaded: ../Features/Features_AdaptiveDEFT_CoAttn_R50_v2/Feature_P01_01_AdaptiveDEFT_CoAttn_R50.npz
Total frames: 36118 | Feature dim: 4096
Unique clips: 307 | Unique actions: 111


## 4. Sliding Window Generator + Clip-Level Split

In [13]:
def build_clip_metadata(features, clip_ids, frame_numbers, action_labels):
    clips = {}
    for cid in np.unique(clip_ids):
        idx = np.where(clip_ids == cid)[0]
        idx = idx[np.argsort(frame_numbers[idx])]
        clips[int(cid)] = {
            'indices':    idx,
            'label':      int(action_labels[idx[0]]),
            'num_frames': len(idx),
        }
    return clips


def windows_from_clips(clip_meta, clip_ids_subset, features, N, stride_ratio=1.0):
    stride = max(1, int(round(N * stride_ratio)))
    windows = []
    for cid in clip_ids_subset:
        meta = clip_meta[cid]
        if meta['num_frames'] < N:
            continue
        for start in range(0, meta['num_frames'] - N + 1, stride):
            sel = meta['indices'][start:start + N]
            windows.append({
                'features':   features[sel],
                'label':      meta['label'],
                'clip_id':    cid,
                'window_idx': start,
            })
    return windows


def clip_level_split(clip_meta, N, seed, val_frac=0.2, test_frac=0.2, min_per_class=2):
    eligible = [cid for cid, m in clip_meta.items() if m['num_frames'] >= N]
    labels = np.array([clip_meta[c]['label'] for c in eligible])
    unique, counts = np.unique(labels, return_counts=True)
    keep = set(unique[counts >= min_per_class].tolist())
    eligible = [c for c, l in zip(eligible, labels) if l in keep]
    labels = np.array([clip_meta[c]['label'] for c in eligible])
    new_unique = sorted(set(labels.tolist()))
    label_remap = {old: new for new, old in enumerate(new_unique)}
    num_classes = len(new_unique)
    if len(eligible) < 4:
        return [], [], [], num_classes, label_remap
    try:
        train_v_cids, test_cids = train_test_split(
            eligible, test_size=test_frac, random_state=seed, stratify=labels)
        train_v_labels = np.array([clip_meta[c]['label'] for c in train_v_cids])
        train_cids, val_cids = train_test_split(
            train_v_cids, test_size=val_frac, random_state=seed, stratify=train_v_labels)
    except ValueError:
        train_v_cids, test_cids = train_test_split(eligible, test_size=test_frac, random_state=seed)
        train_cids, val_cids = train_test_split(train_v_cids, test_size=val_frac, random_state=seed)
    return train_cids, val_cids, test_cids, num_classes, label_remap


def remap_window_labels(windows, label_remap):
    for w in windows:
        w['label'] = label_remap[w['label']]
    return windows


clip_meta = build_clip_metadata(features, clip_ids, frame_numbers, action_labels)
print(f"Clip metadata built: {len(clip_meta)} clips")

Clip metadata built: 307 clips


## 5. SVSG + DPT Graph Builder

In [14]:
def build_window_graph(window_features, percentile=DPT_PERCENTILE):
    x = torch.tensor(window_features, dtype=torch.float32)
    N = x.shape[0]
    sim = F.cosine_similarity(x.unsqueeze(1), x.unsqueeze(0), dim=2)
    edges_found = 0
    p = percentile
    while edges_found == 0 and p > 50:
        thr = torch.quantile(sim, p / 100)
        adj = (sim >= thr).float()
        adj.fill_diagonal_(0)
        rows, cols = torch.nonzero(adj, as_tuple=True)
        edges_found = rows.numel()
        if edges_found == 0:
            p -= 10
    if edges_found == 0:
        sim.fill_diagonal_(-1)
        nn_idx = sim.argmax(dim=1)
        rows = torch.arange(N)
        cols = nn_idx
    edge_index = torch.stack([rows, cols], dim=0).long()
    return Data(x=x, edge_index=edge_index)

## 6. GAT Model + Loss + Dataset

In [15]:
class GATGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, heads=2):
        super().__init__()
        self.conv1 = GATConv(input_dim, hidden_dim, heads=heads)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        g = global_mean_pool(x, batch)
        return F.log_softmax(self.classifier(g), dim=1), x


class MultiTaskGraphLoss(nn.Module):
    def __init__(self, alpha=0.8):
        super().__init__()
        self.ce = nn.NLLLoss()
        self.alpha = alpha
    def forward(self, log_probs, targets, node_feats, batch_idx):
        ce = self.ce(log_probs, targets)
        tloss = 0.0
        ng = batch_idx.max().item() + 1
        for g in range(ng):
            mask = (batch_idx == g)
            f = node_feats[mask]
            if f.shape[0] > 1:
                tloss = tloss + torch.mean(torch.abs(f[1:] - f[:-1]))
        tloss = tloss / max(1, ng)
        return self.alpha * ce + (1 - self.alpha) * tloss


class WindowGraphDataset(torch.utils.data.Dataset):
    def __init__(self, windows, percentile=DPT_PERCENTILE):
        self.windows = windows
        self.percentile = percentile
        self._cache = {}
    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        if idx in self._cache: return self._cache[idx]
        w = self.windows[idx]
        g = build_window_graph(w['features'], self.percentile)
        g.y = torch.tensor([w['label']], dtype=torch.long)
        self._cache[idx] = g
        return g

## 7. Train / Eval 

In [16]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        log_probs, node_feats = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(log_probs, batch.y, node_feats, batch.batch)
        loss.backward(); optimizer.step()
        total += loss.item() * batch.num_graphs
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, num_classes, device):
    model.eval()
    aps, als, aprobs = [], [], []
    for batch in loader:
        batch = batch.to(device)
        lp, _ = model(batch.x, batch.edge_index, batch.batch)
        p = torch.exp(lp); preds = p.argmax(dim=1)
        aps.append(preds.cpu().numpy()); als.append(batch.y.cpu().numpy()); aprobs.append(p.cpu().numpy())
    preds = np.concatenate(aps); labels = np.concatenate(als); probs = np.concatenate(aprobs)
    top1 = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=1)
    rec  = recall_score(labels, preds, average='weighted', zero_division=1)
    f1   = f1_score(labels, preds, average='weighted', zero_division=1)
    k = min(5, num_classes)
    top5_preds = np.argsort(-probs, axis=1)[:, :k]
    top5 = np.mean([y in row for y, row in zip(labels, top5_preds)])
    return top1, top5, prec, rec, f1

## 8. Single Experiment

In [17]:
def run_experiment(N, seed, percentile=DPT_PERCENTILE):
    np.random.seed(seed); torch.manual_seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)

    train_cids, val_cids, test_cids, num_classes, label_remap = clip_level_split(
        clip_meta, N, seed, min_per_class=MIN_PER_CLASS)
    if len(train_cids) == 0: return None

    train_w = windows_from_clips(clip_meta, train_cids, features, N, STRIDE_RATIO)
    val_w   = windows_from_clips(clip_meta, val_cids,   features, N, STRIDE_RATIO)
    test_w  = windows_from_clips(clip_meta, test_cids,  features, N, STRIDE_RATIO)
    train_w = remap_window_labels(train_w, label_remap)
    val_w   = remap_window_labels(val_w,   label_remap)
    test_w  = remap_window_labels(test_w,  label_remap)
    if len(train_w) == 0 or len(test_w) == 0: return None

    train_ds = WindowGraphDataset(train_w, percentile)
    val_ds   = WindowGraphDataset(val_w,   percentile)
    test_ds  = WindowGraphDataset(test_w,  percentile)
    train_loader = PyGDataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = PyGDataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = PyGDataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    model = GATGraphClassifier(features.shape[1], HIDDEN_DIM, num_classes, HEADS).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = MultiTaskGraphLoss(alpha=0.8)

    if device.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    for i in range(len(train_ds)):
        _ = train_ds[i]
    if device.type == "cuda": torch.cuda.synchronize()
    avg_graph_time = (time.time() - t0) / max(1, len(train_ds))

    sg = train_ds[0]
    edges_per_window = sg.edge_index.shape[1]
    total_possible = N * (N - 1)
    sparsity = edges_per_window / total_possible * 100

    t1 = time.time()
    for ep in range(NUM_EPOCHS):
        train_one_epoch(model, train_loader, optimizer, criterion, device)
    if device.type == "cuda": torch.cuda.synchronize()
    train_time = time.time() - t1

    t2 = time.time()
    top1, top5, prec, rec, f1 = evaluate(model, test_loader, num_classes, device)
    if device.type == "cuda": torch.cuda.synchronize()
    infer_time = time.time() - t2

    peak_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0.0

    return {
        "N": N, "Seed": seed, "Num_Classes": num_classes,
        "Train_Clips": len(train_cids), "Val_Clips": len(val_cids), "Test_Clips": len(test_cids),
        "Train_Windows": len(train_w), "Val_Windows": len(val_w), "Test_Windows": len(test_w),
        "Top1_Acc": top1, "Top5_Acc": top5, "Precision": prec, "Recall": rec, "F1": f1,
        "Edges_Per_Window": edges_per_window, "Total_Possible_Edges": total_possible,
        "Sparsity_Percent": sparsity,
        "AvgGraphConstruct_Time_ms": avg_graph_time * 1000,
        "TotalTraining_Time_s": train_time, "Inference_Time_s": infer_time,
        "Peak_GPU_Memory_MB": peak_mem,
    }

## 9. Run All (5 N × 3 seeds = 15 experiments)

In [18]:
all_results = []
for N in N_VALUES:
    print(f"\n{'#'*70}\n# N = {N}\n{'#'*70}")
    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")
        res = run_experiment(N, seed)
        if res is None:
            print(f"  [SKIPPED] insufficient data at N={N}, seed={seed}")
            continue
        all_results.append(res)
        print(f"  [DONE] N={N} seed={seed} | Top1={res['Top1_Acc']:.4f} | F1={res['F1']:.4f} | "
              f"Train={res['Train_Windows']} Test={res['Test_Windows']} | Mem={res['Peak_GPU_Memory_MB']:.1f}MB")

full_df = pd.DataFrame(all_results)
full_df.to_csv(RESULTS_FULL, index=False)
print(f"\n[SAVED] {RESULTS_FULL}")


######################################################################
# N = 30
######################################################################

--- Seed 42 ---
  [DONE] N=30 seed=42 | Top1=0.6089 | F1=0.5866 | Train=569 Test=225 | Mem=48.7MB

--- Seed 123 ---
  [DONE] N=30 seed=123 | Top1=0.3644 | F1=0.3172 | Train=534 Test=225 | Mem=48.7MB

--- Seed 456 ---
  [DONE] N=30 seed=456 | Top1=0.5432 | F1=0.5556 | Train=545 Test=162 | Mem=48.7MB

######################################################################
# N = 100
######################################################################

--- Seed 42 ---
  [DONE] N=100 seed=42 | Top1=0.7400 | F1=0.7061 | Train=107 Test=50 | Mem=75.3MB

--- Seed 123 ---
  [DONE] N=100 seed=123 | Top1=0.7027 | F1=0.6448 | Train=119 Test=37 | Mem=75.3MB

--- Seed 456 ---
  [DONE] N=100 seed=456 | Top1=0.7812 | F1=0.7172 | Train=120 Test=32 | Mem=75.3MB

######################################################################
# N = 150
###########

## 10. Aggregate (Mean ± Std)

In [19]:
metric_cols = ['Top1_Acc', 'Top5_Acc', 'Precision', 'Recall', 'F1',
               'Edges_Per_Window', 'Sparsity_Percent',
               'AvgGraphConstruct_Time_ms', 'TotalTraining_Time_s',
               'Inference_Time_s', 'Peak_GPU_Memory_MB']

agg = full_df.groupby('N')[metric_cols].agg(['mean', 'std']).round(4)
agg.columns = [f"{c}_{s}" for c, s in agg.columns]

first_seed = full_df['Seed'].min()
counts = full_df[full_df['Seed'] == first_seed].set_index('N')[
    ['Num_Classes', 'Train_Clips', 'Val_Clips', 'Test_Clips',
     'Train_Windows', 'Val_Windows', 'Test_Windows']]

summary_df = counts.join(agg).reset_index()
summary_df.to_csv(RESULTS_SUMMARY, index=False)
print(f"[SAVED] {RESULTS_SUMMARY}\n")
summary_df

[SAVED] results_phase3a_NewFeat_R50_summary.csv



,N,Num_Classes,Train_Clips,Val_Clips,Test_Clips,Train_Windows,Val_Windows,Test_Windows,Top1_Acc_mean,Top1_Acc_std,...,Sparsity_Percent_mean,Sparsity_Percent_std,AvgGraphConstruct_Time_ms_mean,AvgGraphConstruct_Time_ms_std,TotalTraining_Time_s_mean,TotalTraining_Time_s_std,Inference_Time_s_mean,Inference_Time_s_std,Peak_GPU_Memory_MB_mean,Peak_GPU_Memory_MB_std
0,30,53,124,32,39,569,105,225,0.5055,0.1265,...,8.9655,0.0,4.3649,0.1800,110.2305,3.5994,1.0424,0.2079,48.6610,0.0000
1,100,13,46,12,15,107,25,50,0.7413,0.0393,...,1.0101,0.0,43.7443,0.6726,26.6192,1.8088,1.9819,0.5982,75.3070,0.0035
2,150,10,36,10,12,68,16,24,0.6655,0.1195,...,1.3423,0.0,120.5235,1.1286,17.7888,1.8013,3.5995,0.4319,105.0092,0.0000
3,200,7,23,6,8,43,9,13,0.5490,0.1433,...,1.5075,0.0,224.8443,1.5750,11.7204,0.9869,3.2929,0.5445,142.4662,0.0068


## 11. Pretty-Printed Table

In [20]:
def fmt(m, s, pct=False, dec=3):
    if pct: return f"{m*100:.{dec-1}f} ± {s*100:.{dec-1}f}"
    return f"{m:.{dec}f} ± {s:.{dec}f}"

print("\n========= PHASE 3a: PER-CLIP WINDOW (NEW R50 FEATURES) =========\n")
print(f"{'N':>4} | {'Classes':>7} | {'Train W':>7} | {'Test W':>6} | "
      f"{'Top-1 (%)':>12} | {'Top-5 (%)':>12} | {'F1':>13} | "
      f"{'GraphTime (ms)':>16} | {'GPU Mem (MB)':>14}")
print("-" * 130)
for _, r in summary_df.iterrows():
    print(f"{int(r['N']):>4} | {int(r['Num_Classes']):>7} | "
          f"{int(r['Train_Windows']):>7} | {int(r['Test_Windows']):>6} | "
          f"{fmt(r['Top1_Acc_mean'], r['Top1_Acc_std'], pct=True):>12} | "
          f"{fmt(r['Top5_Acc_mean'], r['Top5_Acc_std'], pct=True):>12} | "
          f"{fmt(r['F1_mean'], r['F1_std']):>13} | "
          f"{fmt(r['AvgGraphConstruct_Time_ms_mean'], r['AvgGraphConstruct_Time_ms_std'], dec=2):>16} | "
          f"{fmt(r['Peak_GPU_Memory_MB_mean'], r['Peak_GPU_Memory_MB_std'], dec=1):>14}")


========= PHASE 3a: PER-CLIP WINDOW (NEW R50 FEATURES) =========

   N | Classes | Train W | Test W |    Top-1 (%) |    Top-5 (%) |            F1 |   GraphTime (ms) |   GPU Mem (MB)
----------------------------------------------------------------------------------------------------------------------------------
  30 |      53 |     569 |    225 | 50.55 ± 12.65 | 76.47 ± 2.82 | 0.486 ± 0.147 |      4.36 ± 0.18 |     48.7 ± 0.0
 100 |      13 |     107 |     50 | 74.13 ± 3.93 | 90.08 ± 3.63 | 0.689 ± 0.039 |     43.74 ± 0.67 |     75.3 ± 0.0
 150 |      10 |      68 |     24 | 66.55 ± 11.95 | 92.57 ± 7.16 | 0.614 ± 0.120 |    120.52 ± 1.13 |    105.0 ± 0.0
 200 |       7 |      43 |     13 | 54.90 ± 14.33 | 98.04 ± 3.40 | 0.507 ± 0.183 |    224.84 ± 1.57 |    142.5 ± 0.0
